# **Remote MCP Tool Calling** with Responses API (Azure OpenAI)

This notebook demonstrates how to use **remote MCP servers** as tools in the **Responses API** on **Azure OpenAI**. The model can **discover** and **invoke** tools exposed by a remote **MCP server** (here: the **Microsoft Learn MCP Server**) without you implementing a local MCP client; tool execution is handled by the Responses runtime when you declare the MCP server in `tools`. [2](https://platform.openai.com/docs/guides/tools-connectors-mcp)

---

## Goals

- Initialize the `OpenAI` client for **Azure OpenAI** using `base_url = <endpoint>/openai/v1/` [3](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry-classic)  
- Declare a **remote MCP tool** (`type="mcp"`) pointing to **Microsoft Learn MCP Server** (`https://learn.microsoft.com/api/mcp`) [1](https://learn.microsoft.com/en-us/training/support/mcp)  
- Let the model **list** available tools and **use** them (`tool_choice="auto"` or `"required"`) via the Responses API [2](https://platform.openai.com/docs/guides/tools-connectors-mcp)  
- Show **streaming** to observe tool calls in real time [4](https://platform.openai.com/docs/api-reference/responses)  
- Run a simple **multi‑turn** flow with `previous_response_id` while re‑declaring `tools` each turn [5](https://community.openai.com/t/tools-not-working-after-first-turn-when-using-previous-response-id/1368446)

---

## Prerequisites

- Python 3.10+  
- `pip install openai`  
- Azure OpenAI resource with a **model deployment** (e.g., `gpt‑4o-mini`, `gpt‑4.1-mini`, `gpt‑5.1-chat`, etc.) and **Responses API v1** access (use `base_url = <endpoint>/openai/v1/`) [3](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry-classic)  
- Environment variables:
  - `AZURE_OPENAI_API_KEY`
  - `AZURE_OPENAI_ENDPOINT` (e.g., `https://<resource>.openai.azure.com`)
  - `MODEL_DEPLOYMENT_NAME` (your Azure model **deployment name**)

> ℹ️ **About remote MCP servers**  
> The Responses API can call **remote MCP servers** using the built‑in `mcp` tool type. You provide the `server_url` (and optional headers/approval policy), and the model lists and invokes server tools directly. **Trust the server you connect to**; a malicious server could leak sensitive data. This notebook uses the **Microsoft Learn MCP Server**, which is public and does **not** require auth. [2](https://platform.openai.com/docs/guides/tools-connectors-mcp)[1](https://learn.microsoft.com/en-us/training/support/mcp)


# Constants and Libraries

In [1]:
import os, json
from pprint import pprint
from IPython.display import Markdown, Image, display
from dotenv import load_dotenv
from openai import AzureOpenAI

if not load_dotenv("./../../config/credentials_my.env"):
    print("Environment variables not loaded, cell execution stopped")
else:
    print("Environment variables have been loaded ;-)")

MODEL = os.environ["MODEL_DEPLOYMENT_NAME"]

Environment variables have been loaded ;-)


In [2]:
# Initialize client
client = AzureOpenAI(
    api_version = os.getenv("AZURE_OPENAI_API_VERSION")
    # api_key = os.getenv("AZURE_OPENAI_API_KEY"),
    # azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
)

print(f"OpenAI Endpoint base_url: {client.base_url}")

OpenAI Endpoint base_url: https://mmoaiswc-01.openai.azure.com/openai/


In [3]:
# MCP-specific configuration

# --- Remote MCP server: Microsoft Learn MCP Server (Streamable HTTP) ---
# Public endpoint; no auth required for tool discovery/usage
MSLEARN_MCP_URL = "https://learn.microsoft.com/api/mcp"  # streamable HTTP MCP endpoint

mcp_tool_mslearn = {
    "type": "mcp",
    "server_label": "mslearn",
    "server_description": "Microsoft Learn MCP server with docs/tools.",
    "server_url": MSLEARN_MCP_URL,
    # for demo flows, remove manual approvals
    "require_approval": "never",
    # If a server required auth, you would add headers here, for example:
    # "headers": {"Authorization": "Bearer <TOKEN>"}
}
mcp_tool_mslearn

{'type': 'mcp',
 'server_label': 'mslearn',
 'server_description': 'Microsoft Learn MCP server with docs/tools.',
 'server_url': 'https://learn.microsoft.com/api/mcp',
 'require_approval': 'never'}

# List available tools from the MCP server

In [4]:
# Force the model to use the MCP tool to list server tools

resp_list = client.responses.create(
    model=MODEL,
    tools=[mcp_tool_mslearn],
    tool_choice="required",
    input="List the tools.",
    store=True,
)

display(Markdown(resp_list.output_text))

Here's a list of tools available for working with Azure:

### Azure Command-Line Tools
1. **Azure CLI**: Command-line tool to manage Azure resources.
2. **Azure PowerShell**: PowerShell module for managing Azure resources directly.
3. **Azure Developer CLI (`azd`)**: Accelerates application deployment from local development to Azure.
4. **AzCopy**: Command-line tool for data transfer to and from Azure Storage.

### GUI Tools for Azure Storage
1. **Azure Storage Explorer**: Manage storage resources; supports Windows, macOS, and Linux.
2. **Azure Data Studio**: Cross-platform tool for managing SQL databases.

### Development Tools
1. **Azure Functions**: Serverless compute service to run responsive applications.
2. **Azure Kubernetes Service**: Manages Kubernetes clusters.

### AI and Machine Learning Tools
1. **Azure AI Foundry**: Work with models and deployments.
2. **Azure AI Search**: Manage search resources and queries.

### Analytics Tools
1. **Azure Data Explorer**: Analyze large volumes of data interactively.
2. **Azure Event Hubs**: Process large data streams in real time.

### Management and Governance Tools
1. **Azure Monitor**: Logs and metrics for monitoring Azure resources.
2. **Azure Managed Grafana**: Observability platform for managing Grafana workspaces.

### Container Services
1. **Azure Container Instances**: Deploy containers with ease without managing servers.

These tools help enhance productivity, management, and analysis across various Azure services. For further details, visit the official [Azure Tools Documentation](https://learn.microsoft.com/en-us/azure/developer/azure-mcp-server/tools).

# Use MCP tools: query Microsoft Learn docs

# Streaming to observe MCP tool calls

In [5]:
query = (
    "Using the MCP tools, find Microsoft Learn documentation about the Azure OpenAI Responses API,"
    "then summarize key steps to use the v1 endpoint on Azure "
    "(base_url, deployment name) with short bullet points and include links."
)

resp_query = client.responses.create(
    model=MODEL,
    tools=[mcp_tool_mslearn],
    tool_choice="auto",  # let the model decide which MCP tool(s) to use
    input=query,
    store=True,
)

display(Markdown(resp_query.output_text))

Here are the key steps to use the Azure OpenAI Responses API v1 endpoint on Azure:

- **Set the Base URL**:
  - Format: `https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/`
  - Replace `YOUR-RESOURCE-NAME` with your specific resource name.

- **Deployment Name**:
  - Use the model deployment name, such as `gpt-4.1-nano`.

- **Authentication**:
  - Use either API key or token-based authentication.
  - **API Key**: Store securely in Azure Key Vault.
  - **Authorization Token**: Generate via Azure CLI.

- **Making API Requests**:
  - Use HTTP POST to create a response:
    ```bash
    curl -X POST https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/responses \
      -H "Content-Type: application/json" \
      -H "Authorization: Bearer $AZURE_OPENAI_AUTH_TOKEN" \
      -d '{
         "model": "gpt-4.1-nano",
         "input": "This is a test"
        }'
    ```

- **Retrieving Responses**:
  - Use HTTP GET to retrieve a response:
    ```bash
    curl -X GET https://YOUR-RESOURCE-NAME.openai.azure.com/openai/v1/responses/{response_id} \
      -H "Content-Type: application/json" \
      -H "Authorization: Bearer $AZURE_OPENAI_AUTH_TOKEN"
    ```

#### Useful Links
- [Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry-classic)
- [API Keys with Azure Key Vault](https://learn.microsoft.com/en-us/azure/key-vault/general/apps-api-keys-secrets)

These steps help you set up and interact with the Azure OpenAI Responses API effectively.

## Operational notes

- **Pass `tools` on every turn**: tool availability is **not** carried automatically with `previous_response_id`. Re‑declare the `mcp` tool each request to keep tools active. [5](https://community.openai.com/t/tools-not-working-after-first-turn-when-using-previous-response-id/1368446)  
- **Endpoint path on Azure**: use `base_url = <endpoint>/openai/v1/` to access the latest Responses API surface with the OpenAI SDK against Azure OpenAI. [3](https://learn.microsoft.com/en-us/azure/ai-foundry/openai/how-to/responses?view=foundry-classic)  
- **Remote server trust**: only connect to trusted MCP servers; a malicious server could exfiltrate sensitive context. Microsoft Learn MCP is public and does not require auth. [2](https://platform.openai.com/docs/guides/tools-connectors-mcp)[1](https://learn.microsoft.com/en-us/training/support/mcp)  
- **Transport**: modern MCP servers use **Streamable HTTP** (preferred) vs. legacy SSE; Learn MCP uses streamable HTTP. [6](https://blog.fka.dev/blog/2025-06-06-why-mcp-deprecated-sse-and-go-with-streamable-http/)[1](https://learn.microsoft.com/en-us/training/support/mcp)
